# 09 - WEO Excel Visualization and EDA Lab

This lab uses the WEO Excel workbook to create graphs for different economic scenarios. You will build analysis-ready DataFrames, then draw line charts, scatter plots, histograms, box plots, horizontal bars, heatmaps, and a small dashboard.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server query mode.

This notebook uses the WEO Excel workbook from Module 4. Locally, it looks for `Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx`. In Google Colab, upload `WEOApr2026all.xlsx` into the session files panel or use the upload prompt in the setup cell.

## 0. Setup

In [1]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn openpyxl joblib -q

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def find_weo_workbook():
    """Find the WEO workbook whether the notebook runs from the lab folder, repo root, or Colab."""
    candidate_paths = [
        Path("../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("data/WEOApr2026all.xlsx"),
        Path("/content/WEOApr2026all.xlsx"),
        Path("WEOApr2026all.xlsx"),
    ]
    for path in candidate_paths:
        if path.exists():
            return path

    # Colab fallback: ask the learner to upload the Excel file if it is missing.
    try:
        from google.colab import files
        print("Upload WEOApr2026all.xlsx from Module 4 to continue.")
        uploaded = files.upload()
        if "WEOApr2026all.xlsx" in uploaded:
            return Path("WEOApr2026all.xlsx")
    except Exception:
        pass

    raise FileNotFoundError(
        "WEOApr2026all.xlsx was not found. Run locally from the repo, or upload the Excel file in Google Colab."
    )


WEO_PATH = find_weo_workbook()
print("Using workbook:", WEO_PATH)

excel_file = pd.ExcelFile(WEO_PATH)
print("Workbook sheets:", excel_file.sheet_names)

Note: you may need to restart the kernel to use updated packages.
Using workbook: ../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx
Workbook sheets: ['April 2026 WEO', 'Countries', 'Country Groups', 'Commodity Prices', 'Country Group Composition']


## 1. Build WEO DataFrames

We use the Countries sheet for country-level analysis, the Country Groups sheet for regional/world trends, and the Commodity Prices sheet for commodity scenarios.

In [ ]:
INDICATOR_LABELS = {
    "NGDP_RPCH": "gdp_growth_pct",
    "PCPIPCH": "inflation_pct",
    "LUR": "unemployment_rate",
    "BCA_NGDPD": "current_account_pct_gdp",
    "GGXWDG_NGDP": "government_debt_pct_gdp",
    "NGDPDPC": "gdp_per_capita_usd",
    "NID_NGDP": "investment_pct_gdp",
    "NGSD_NGDP": "savings_pct_gdp",
    "TX_RPCH": "export_volume_growth_pct",
    "TM_RPCH": "import_volume_growth_pct",
}


def get_year_columns(frame):
    """Return Excel columns that represent years such as 1980, 2024, or 2031."""
    return [column for column in frame.columns if isinstance(column, int) or str(column).isdigit()]


def load_weo_sheets(path):
    """Load the useful WEO workbook sheets into separate DataFrames."""
    countries = pd.read_excel(path, sheet_name="Countries")
    country_groups = pd.read_excel(path, sheet_name="Country Groups")
    commodity_prices = pd.read_excel(path, sheet_name="Commodity Prices")
    group_composition = pd.read_excel(path, sheet_name="Country Group Composition")
    return countries, country_groups, commodity_prices, group_composition


def make_long_indicator_data(frame, indicator_ids, source_name):
    """Convert WEO wide year columns into a tidy row-per-country-year format."""
    year_columns = get_year_columns(frame)
    id_columns = ["COUNTRY.ID", "COUNTRY", "INDICATOR.ID", "INDICATOR", "UNIT"]
    filtered = frame.loc[frame["INDICATOR.ID"].isin(indicator_ids), id_columns + year_columns].copy()

    long_df = filtered.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name="year",
        value_name="value",
    )
    long_df["year"] = long_df["year"].astype(int)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df["source_sheet"] = source_name
    return long_df


def make_country_macro_dataframe(countries, group_composition):
    """Create one country-year row with selected WEO indicators as columns."""
    long_df = make_long_indicator_data(countries, list(INDICATOR_LABELS), "Countries")

    macro = (
        long_df.pivot_table(
            index=["COUNTRY.ID", "COUNTRY", "year"],
            columns="INDICATOR.ID",
            values="value",
            aggfunc="first",
        )
        .reset_index()
        .rename(columns=INDICATOR_LABELS)
    )

    advanced_codes = set(group_composition.loc[group_composition["groupname"].eq("Advanced Economies"), "countrycode"])
    emerging_codes = set(group_composition.loc[group_composition["groupname"].eq("Emerging Market and Developing Economies"), "countrycode"])
    ssa_codes = set(group_composition.loc[group_composition["groupname"].eq("Sub-Saharan Africa (SSA)"), "countrycode"])

    macro["economic_group"] = np.select(
        [macro["COUNTRY.ID"].isin(advanced_codes), macro["COUNTRY.ID"].isin(emerging_codes)],
        ["Advanced Economies", "Emerging Market and Developing Economies"],
        default="Other",
    )
    macro["is_sub_saharan_africa"] = macro["COUNTRY.ID"].isin(ssa_codes)
    return macro, long_df


countries_raw, country_groups_raw, commodity_prices_raw, group_composition = load_weo_sheets(WEO_PATH)
country_macro, country_long = make_country_macro_dataframe(countries_raw, group_composition)

print("Countries sheet rows:", len(countries_raw))
print("Country-year analytical rows:", len(country_macro))
display(country_macro.head())


def make_group_long(country_groups):
    indicator_ids = ["NGDP_RPCH", "PCPIPCH", "TM_RPCH", "TX_RPCH"]
    return make_long_indicator_data(country_groups, indicator_ids, "Country Groups")


def make_commodity_long(commodity_prices):
    indicator_ids = ["POILAPSP", "PFOODW", "PNGASW", "PCOALW", "PALLFNFW"]
    return make_long_indicator_data(commodity_prices, indicator_ids, "Commodity Prices")


group_long = make_group_long(country_groups_raw)
commodity_long = make_commodity_long(commodity_prices_raw)
print("Group trend rows:", len(group_long))
print("Commodity trend rows:", len(commodity_long))

## 2. Scenario 1: GDP Growth Trend by Economic Group

A line chart is best when the x-axis is time. Here we compare world and regional GDP growth trends.

In [ ]:
trend_groups = ["World", "Advanced Economies", "Emerging Market and Developing Economies", "Sub-Saharan Africa (SSA)"]
growth_trend = group_long[
    group_long["COUNTRY"].isin(trend_groups)
    & group_long["INDICATOR.ID"].eq("NGDP_RPCH")
    & group_long["year"].between(2015, 2031)
].copy()

plt.figure(figsize=(11, 5))
sns.lineplot(data=growth_trend, x="year", y="value", hue="COUNTRY", marker="o")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Real GDP Growth Trend by WEO Group")
plt.xlabel("Year")
plt.ylabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_group_gdp_growth_trend.png", dpi=150)
plt.show()

## 3. Scenario 2: Top and Bottom GDP Growth Countries

Horizontal bars work well for ranked country lists.

In [ ]:
analysis_year = 2026
latest = country_macro[country_macro["year"].eq(analysis_year)].copy()
ranked_growth = latest.dropna(subset=["gdp_growth_pct"]).sort_values("gdp_growth_pct")
plot_ranked = pd.concat([ranked_growth.head(10), ranked_growth.tail(10)])

plt.figure(figsize=(9, 8))
sns.barplot(data=plot_ranked, x="gdp_growth_pct", y="COUNTRY", hue="economic_group", dodge=False)
plt.axvline(0, color="black", linewidth=0.8)
plt.title(f"Top and Bottom Real GDP Growth Countries, {analysis_year}")
plt.xlabel("GDP Growth (%)")
plt.ylabel("Country")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_top_bottom_growth_bar.png", dpi=150)
plt.show()

## 4. Scenario 3: Inflation vs GDP Growth Scatter Plot

Each dot is a country. Color shows whether it is an advanced economy or an emerging/developing economy.

In [ ]:
scatter_df = latest.dropna(subset=["inflation_pct", "gdp_growth_pct"]).copy()

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=scatter_df,
    x="inflation_pct",
    y="gdp_growth_pct",
    hue="economic_group",
    size="gdp_per_capita_usd",
    sizes=(30, 220),
    alpha=0.75,
)
plt.axhline(0, color="black", linewidth=0.8)
plt.title(f"Inflation vs GDP Growth by Country, {analysis_year}")
plt.xlabel("Inflation (%)")
plt.ylabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_inflation_vs_growth_scatter.png", dpi=150)
plt.show()

## 5. Scenario 4: Inflation Distribution

A histogram shows how inflation values are distributed across countries.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(latest["inflation_pct"].dropna(), bins=20, edgecolor="black")
plt.title(f"Distribution of Country Inflation Rates, {analysis_year}")
plt.xlabel("Inflation (%)")
plt.ylabel("Number of Countries")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_inflation_histogram.png", dpi=150)
plt.show()

## 6. Scenario 5: Box Plot by Economic Group

A box plot compares spread, median, and potential outliers across groups.

In [ ]:
box_df = latest[latest["economic_group"].isin(["Advanced Economies", "Emerging Market and Developing Economies"])].copy()

plt.figure(figsize=(9, 4))
sns.boxplot(data=box_df, x="economic_group", y="gdp_growth_pct")
plt.title(f"GDP Growth Spread by Economic Group, {analysis_year}")
plt.xlabel("Economic Group")
plt.ylabel("GDP Growth (%)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_growth_boxplot_by_group.png", dpi=150)
plt.show()

## 7. Scenario 6: Correlation Heatmap

A heatmap is useful when comparing several numeric indicators at once.

In [ ]:
heatmap_columns = [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_rate",
    "government_debt_pct_gdp",
    "current_account_pct_gdp",
    "investment_pct_gdp",
    "savings_pct_gdp",
    "gdp_per_capita_usd",
]

corr = latest[heatmap_columns].corr()
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title(f"WEO Indicator Correlation Heatmap, {analysis_year}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_indicator_correlation_heatmap.png", dpi=150)
plt.show()

## 8. Scenario 7: Commodity Price Trends

Commodity prices are stored in a separate workbook sheet. This chart compares oil, food, natural gas, coal, and the all-commodity index.

In [ ]:
commodity_names = {
    "POILAPSP": "Oil price",
    "PFOODW": "Food index",
    "PNGASW": "Natural gas index",
    "PCOALW": "Coal index",
    "PALLFNFW": "All commodities index",
}
commodity_plot = commodity_long[commodity_long["year"].between(2015, 2031)].copy()
commodity_plot["indicator_name"] = commodity_plot["INDICATOR.ID"].map(commodity_names)

plt.figure(figsize=(11, 5))
sns.lineplot(data=commodity_plot, x="year", y="value", hue="indicator_name", marker="o")
plt.title("Selected Commodity Price Trends")
plt.xlabel("Year")
plt.ylabel("Value / Index")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_commodity_price_trends.png", dpi=150)
plt.show()

## 9. Scenario Dashboard

A dashboard places related charts in one figure for quick review.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

sns.lineplot(data=growth_trend, x="year", y="value", hue="COUNTRY", ax=axes[0, 0], legend=False)
axes[0, 0].set_title("GDP Growth Trend")
axes[0, 0].set_xlabel("Year")
axes[0, 0].set_ylabel("Growth (%)")

sns.scatterplot(data=scatter_df, x="inflation_pct", y="gdp_growth_pct", hue="economic_group", ax=axes[0, 1], legend=False)
axes[0, 1].set_title("Inflation vs Growth")
axes[0, 1].set_xlabel("Inflation (%)")
axes[0, 1].set_ylabel("Growth (%)")

sns.histplot(latest["inflation_pct"].dropna(), bins=20, ax=axes[1, 0])
axes[1, 0].set_title("Inflation Distribution")
axes[1, 0].set_xlabel("Inflation (%)")

sns.boxplot(data=box_df, x="economic_group", y="gdp_growth_pct", ax=axes[1, 1])
axes[1, 1].set_title("Growth by Group")
axes[1, 1].set_xlabel("")
axes[1, 1].set_ylabel("Growth (%)")
axes[1, 1].tick_params(axis="x", rotation=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_eda_dashboard.png", dpi=150)
plt.show()